In [ ]:
import pandas as pd
import numpy as np
import gc

# 기존 parquet 데이터 로드
df_trans = pd.read_parquet(r'../Data Folder\H&M dataset\H&m parquet dataset\transactions.parquet')
df_cust = pd.read_parquet(r'../Data Folder\H&M dataset\H&m parquet dataset\customers.parquet')
df_art = pd.read_parquet(r'../Data Folder\H&M dataset\H&m parquet dataset\articles.parquet')

# ------------------------------------------------------------------------------------------------------------# 
# Customer 데이터 전처리

# [근거 1] FN, Active 결측치 0으로 채우기 및 조건부 1 부여
df_cust['fashion_news_frequency'] = df_cust['fashion_news_frequency'].fillna('NONE').astype(str)
df_cust.loc[(df_cust['FN'].isna()) & (df_cust['fashion_news_frequency'].isin(['Regularly', 'Monthly'])), 'FN'] = 1
df_cust['FN'] = df_cust['FN'].fillna(0).astype(np.int8)
df_cust['Active'] = df_cust['Active'].fillna(0).astype(np.int8)

# 클럽 멤버 상태 결측치 처리
# ---> 기존 카테고리 제한을 object로 잠시 푼 뒤, UNKNOWN으로 채우고 다시 압축했습니다.
df_cust['club_member_status'] = df_cust['club_member_status'].astype(object).fillna('UNKNOWN').astype('category')

# [근거 2] 나이(Age) 결측치 처리 (삭제 -> 중앙값 대체)
age_median = df_cust[df_cust['age'] != -1]['age'].median()
df_cust['age'] = df_cust['age'].replace(-1, age_median).fillna(age_median).astype(np.int8)

# [근거 3] 연령대(Age Group) 파생 변수 생성
bins = [0, 20, 30, 40, 50, 60, 150]
labels = ['10대', '20대', '30대', '40대', '50대', '60대 이상']
df_cust['age_group'] = pd.cut(df_cust['age'], bins=bins, labels=labels, right=False).astype('category')

# postal_code 컬럼 제거
df_cust.drop(columns=['postal_code'], inplace=True, errors='ignore')

# Articles 테이블 전처리 (불필요 컬럼 제거)
cols_to_drop = [
    'detail_desc', 'product_code', 'product_type_no', 'graphical_appearance_no', 
    'colour_group_code', 'perceived_colour_value_id', 'perceived_colour_master_id', 
    'department_no', 'index_code', 'index_group_no', 'section_no', 'garment_group_no'
]
df_art.drop(columns=cols_to_drop, inplace=True, errors='ignore')


# -------------------------------------------------------------------------------------------------------------# 
# Transactions 테이블 전처리 (이상치 제거 및 가격 변환)

# [근거 4] 가격 원화 환산
df_trans['price'] = np.round(df_trans['price'] * 590, 2).astype(np.float32)

# [근거 5] 이상치 제거: 상위 0.1% 대량 구매자(리셀러/봇)으로 판단 필터링
buy_counts = df_trans.groupby('customer_id').size()
outlier_threshold = buy_counts.quantile(0.999) # 상위 0.1% 기준선
normal_customers = buy_counts[buy_counts <= outlier_threshold].index

# 정상 고객만 남기기
df_trans = df_trans[df_trans['customer_id'].isin(normal_customers)]

# -------------------------------------------------------------------------------------------------------------# 
# 최종 병합 (merge처리) 및 저장

# Transaction 기준으로 병합 (on=Left Join)
df_merge = df_trans.merge(df_cust, on='customer_id', how='left')
df_merge = df_merge.merge(df_art, on='article_id', how='left')

# 메모리 최적화를 위해 문자열이었던 ID들을 다시 유지하되 범주형으로 압축.
df_merge.to_parquet('V2_merge.parquet', index=False)

# 메모리 청소
del df_trans, df_cust, df_art, df_merge
gc.collect() ## 메모리 확인용

979